In [ ]:
! pip install PySpice

In [ ]:
import spiceypy as sp
import numpy as np

import os
entries = os.listdir("../../../spice/kernels/mk/")
print(entries)

body_name = "DIMORPHOS"
body_name = "DIMORPHOS"

path = os.path.abspath("../../../spice/kernels/mk/hera_ops.tm")
if os.path.exists(path):
    print("Path exists")
else:
    print("Path does not exist")

os.chdir(os.path.dirname(path))
sp.furnsh(path)

radii = sp.bodvrd(body_name, "RADII", 3)[1]  # returns array [a, b, c] (km)
mean_radius_km = float(np.mean(radii))
print("radies (km):", radii)

# 3) Convert input geodetic lon/lat (deg) to radians; assume planetocentric lat, east-longitude
def dimorphos_ll_to_xyzApprox(lon_deg: float, lat_deg: float):
    lon = np.deg2rad(lon_deg)
    lat = np.deg2rad(lat_deg)
    # For spherical approximation: SPICE latrec(r, lon, lat) -> (x, y, z) in body-fixed frame
    x, y, z = sp.latrec(mean_radius_km, lon, lat)
    return np.array([x, y, z])  # km in DIMORPHOS-fixed frame

re = radii[0]  # equatorial radius (km)
rp = radii[2]  # polar radius (km)
f  = (re - rp) / re  # flattening factor


def dimorphos_ll_to_xyz_pgr(lon_deg, lat_deg, alt_km=0.0):
    lon = np.deg2rad(lon_deg)
    lat = np.deg2rad(lat_deg)
    x, y, z = sp.pgrrec(body_name, lon, lat, alt_km, re, f)
    return np.array([x, y, z])


# Example
#xyz_bf = dimorphos_ll_to_xyz(lon_deg=45.0, lat_deg=10.0)
#print("Body-fixed XYZ (km):", xyz_bf)
xyz_bf = dimorphos_ll_to_xyzApprox(lon_deg=45.0, lat_deg=10.0)
print("Body-fixed XYZ (km):", xyz_bf)

# 4) Rotate to J2000 at a given epoch
et = sp.str2et("2025-03-12T17:41:00Z")
mx2 = sp.pxform("IAU_MARS", "J2000", et)
print(mx2)

# 4) Optional: rotate to J2000 at a given epoch
et = sp.str2et("2025-03-12T17:41:00Z")
#mx = sp.pxform("DIMORPHOS_FIXED", "Didymos_FIXED", et)
#xyz_j2000 = mx @ xyz_bf
#print("J2000 XYZ (km):", xyz_j2000)

# 5) Cleanup if desired
# sp.unload("dimorphos.tm")